# Figure 4 of *A velocity-sorting obstruction for Array-RQMC*
## The condensate: what the sorted rule's stationary state actually looks like

**Purpose:** Regenerate Figure 4 of the manuscript from scratch, and check the
claims the figure is used to support.

**Companion text:** J. M. Hyman, *A velocity-sorting obstruction for Array-RQMC*
(`hyman2026velocity`).
**Section covered:** §6, "What the stationary state looks like".
**Figure reproduced:** Figure 4 (`fig:condensate`).

**Key claims tested:**
- The random-pairing control is state independent and must return the exact
  equilibrium on the sphere $\sum_i v_i^2 = N$. If it does not, nothing else
  in the figure can be believed.
- The sorted rule concentrates almost all mass at the origin and empties the
  shoulder.
- Its survival function drops to a plateau near $2/N$ and holds it out to
  $|v| = \sqrt{N}$, where it stops.
- The apparent contradiction is not one: the sorted rule retains about
  $2.1\%$ of the equilibrium's two-sigma mass while its fourth and sixth
  moments are inflated by factors of order $N/4$ and $N^2/24$.

**Baseline from prior work:** the registered run, seeds 813950001 and
813950002 at $N = 2048$, gives $m_4 = 1545$ and $1530$, $m_6 = 2.650\times10^6$
and $2.604\times10^6$, and $P(|v|>2) = 0.00095$ for both seeds. The control
gives $m_4 = 2.996$ and $3.001$ against an exact $2.9971$.

**Why this notebook exists.** The manuscript's PDF figure is generated by
`kac_fig4_condensate_v1_0_0.py`. A PNG copy was previously kept alongside it
but was withdrawn: the hosting environment stamps C2PA metadata into PNG
files, which changes the file hash without touching a single pixel and makes
an uncorrupted figure read as corrupt. Regenerate here instead.

In [ ]:
# ── Dependencies ──────────────────────────────────────────────────────────────
"""
Figure 4 of hyman2026velocity: the condensate of the sorted Kac rule.
Companion text §6; reproduces Figure 4 (fig:condensate).

Author:  James M. Hyman
         Department of Mathematics, Tulane University
         mhyman@tulane.edu
Date:    2026-09-05  Version 1.0
"""
import base64
import io
import warnings
from datetime import datetime
from math import erfc, factorial, sqrt

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

warnings.filterwarnings('ignore')

# ── Reproducibility ───────────────────────────────────────────────────────────
# The manuscript figure pools two seeds. They are the registered values; changing
# them changes the figure, which is why they are named constants and not literals.
SEEDS = (813950001, 813950002)
np.random.seed(SEEDS[0])          # for any code using the legacy interface

# ── Plot style ────────────────────────────────────────────────────────────────
mpl.rcParams.update({
    'font.size'      : 12,
    'axes.labelsize' : 13,
    'axes.titlesize' : 13,
    'legend.fontsize': 11,
    'figure.dpi'     : 120,
    'lines.linewidth': 1.8,
})

# ── Scale switch ──────────────────────────────────────────────────────────────
# FULL = False  →  quick verification run, well under a minute; the shape of the
#                  figure is already correct but the wings are ragged.
# FULL = True   →  the registered run that produced the manuscript figure.
#                  About 90 s on a Colab CPU. No GPU is used or needed.
FULL = False

SCALE = dict(
    n_part   = 2048,
    n_chains = 50,
    burn     = 2000 if FULL else 200,
    collect  = 40 if FULL else 8,
    gap      = 25 if FULL else 10,
)
print(f"Scale: {'FULL (reproduces the manuscript figure)' if FULL else 'QUICK (verification)'}")
print(f"Parameters: {SCALE}")
print(f"Steps per arm per seed: {SCALE['burn'] + SCALE['collect'] * SCALE['gap']}")

## Background (§2, §6)

The Kac walk holds $N$ velocities on the sphere

$$\sum_{i=1}^{N} v_i^2 = N,$$

and at each step picks a pair $(i,j)$ and rotates it by a uniform angle,

$$v_i' = v_i\cos\theta + v_j\sin\theta, \qquad
  v_j' = -v_i\sin\theta + v_j\cos\theta .$$

The rotation preserves $v_i^2 + v_j^2$ exactly, so the walk never leaves the
sphere and no projection step is needed.

Two pairing rules are compared.

**Random pairing** chooses the pair uniformly and independently of the state.
It is therefore a legitimate Kac step and its stationary measure is the uniform
measure on the sphere. This is the *null control*: it must return the known
equilibrium, and if it does not, the sorted arm tells us nothing.

**The sorted rule** sorts by $|v|$ and pairs adjacent ranks, so the two slowest
particles are paired together, the next two together, and so on. This is the
velocity-sorting heuristic the manuscript studies. It is state dependent, and
that is precisely the hypothesis the Kac equilibrium argument needs.

The exact even moments of the equilibrium on the sphere are

$$m_{2k} \;=\; \frac{N^k\,(2k-1)!!}{N(N+2)\cdots(N+2k-2)},$$

which tend to the Maxwellian values $3$ and $15$ as $N \to \infty$. At
$N = 2048$ they are $2.9971$ and $14.956$, and the difference from $3$ and $15$
is already below the precision this notebook quotes.

> **Departure from the manuscript.** The manuscript's generator writes a PDF for
> LaTeX. This notebook draws the same figure on screen and additionally emits a
> self-contained HTML report.

In [ ]:
# ── Verification Suite ────────────────────────────────────────────────────────
# Every formula used below is checked against an analytically known case before
# any simulation runs. A failure here halts the notebook before wasting compute.
print("=" * 65)
print("VERIFICATION SUITE")
print("=" * 65)


def dfact(n: int) -> int:
    """Double factorial (2k-1)!! for odd argument n = 2k-1."""
    r = 1
    for i in range(n, 0, -2):
        r *= i
    return r


def sphere_moment(N: int, k: int) -> float:
    """Exact 2k-th moment of the uniform measure on sum v_i^2 = N."""
    num, den = N ** k * dfact(2 * k - 1), 1
    for j in range(k):
        den *= (N + 2 * j)
    return num / den


# Test 1: double factorial against hand values 1!!=1, 3!!=3, 5!!=15.
assert (dfact(1), dfact(3), dfact(5)) == (1, 3, 15), "FAIL: dfact"
print("Test 1 PASS  (2k-1)!! = 1, 3, 15 for k = 1, 2, 3")

# Test 2: the sphere moments must tend to the Maxwellian values 3 and 15.
m4_big, m6_big = sphere_moment(10 ** 7, 2), sphere_moment(10 ** 7, 3)
assert abs(m4_big - 3.0) < 1e-4 and abs(m6_big - 15.0) < 1e-3, "FAIL: large-N limit"
print(f"Test 2 PASS  sphere moments -> Maxwellian: m4 = {m4_big:.6f}, m6 = {m6_big:.5f}")

# Test 3: the rotation must conserve pair energy to machine precision.
_rng = np.random.default_rng(0)
_a, _b = _rng.normal(size=2)
_t = 2 * np.pi * _rng.random()
_r = np.hypot(_a, _b)
_new = (_r * np.cos(_t), _r * np.sin(_t))
assert abs(_new[0] ** 2 + _new[1] ** 2 - (_a ** 2 + _b ** 2)) < 1e-12, "FAIL: rotation"
print(f"Test 3 PASS  pair energy conserved to {abs(_new[0]**2 + _new[1]**2 - (_a**2 + _b**2)):.2e}")

# Test 4: two-sided Gaussian tail P(|v| > 2) = erfc(2/sqrt 2).
TAIL2_EXACT = erfc(2.0 / sqrt(2.0))
assert abs(TAIL2_EXACT - 0.0455002639) < 1e-9, "FAIL: tail mass"
print(f"Test 4 PASS  P(|v| > 2) = {TAIL2_EXACT:.10f}")

N_TESTS = 4
print(f"\nAll {N_TESTS} verification tests PASSED.")
print("=" * 65)

## Implementation (§6)

One function does the work. `step_sphere` is reproduced **byte-identically** from the
manuscript's registered generator `kac_fig4_condensate_v1_0_0.py`, which reproduces
it byte-identically from `kac_manuscript_data_v1_0_0.py`, so this notebook and the
published figure share a single implementation of the dynamics rather than two that
might drift apart. Nothing is added to the function, not even a docstring, so the
claim can be checked by string comparison instead of by eye; its documentation sits
in a comment above it. Verification cell 6 asserts the identity at run time.


In [ ]:
# ── The Kac step under both pairing rules  (§2, §6) ──────────────────────────
#
# `step_sphere` below is reproduced BYTE-IDENTICALLY from the registered
# generator kac_fig4_condensate_v1_0_0.py, which in turn reproduces it
# byte-identically from kac_manuscript_data_v1_0_0.py. Nothing is added to it,
# not even a docstring or a type annotation, so the claim is checkable by plain
# string comparison rather than by eye. Its documentation lives here instead.
#
#   v     ndarray, shape (n_chains, n_part). Velocities. Each row lies on the
#         sphere sum_i v_i^2 = n_part and stays there, because the rotation
#         below conserves each pair's energy exactly.
#   rule  'sorted' pairs by ascending |v|, the velocity-sorting heuristic;
#         'random' pairs uniformly at random, the state-independent null
#         control; 'fixed' pairs adjacent indices. This notebook calls only the
#         first two, but the branch is kept so the copy is complete.
#   rng   numpy.random.Generator.
#   ->    the same array, advanced one step, in place.
#
# Pairing is by adjacent rank: after the sort, rank 0 pairs with rank 1, rank 2
# with rank 3, and so on. Under 'sorted' this puts the two slowest particles
# together and the two fastest together.
def step_sphere(v, rule, rng):
    n_chains, n_part = v.shape
    if rule == "sorted":
        o = np.argsort(np.abs(v), axis=1, kind="stable")
    elif rule == "random":
        o = np.argsort(rng.random((n_chains, n_part)), axis=1)
    elif rule == "fixed":
        o = np.tile(np.arange(n_part), (n_chains, 1))
    else:
        raise ValueError(rule)
    ia, ib = o[:, 0::2], o[:, 1::2]
    va = np.take_along_axis(v, ia, axis=1)
    vb = np.take_along_axis(v, ib, axis=1)
    r = np.hypot(va, vb)
    t = 2.0 * np.pi * rng.random(r.shape)
    np.put_along_axis(v, ia, r * np.cos(t), axis=1)
    np.put_along_axis(v, ib, r * np.sin(t), axis=1)
    return v


def sample(rule: str, seed: int) -> np.ndarray:
    """Burn in, then pool decorrelated snapshots of every velocity component."""
    n_part, n_chains = SCALE["n_part"], SCALE["n_chains"]
    rng = np.random.default_rng(seed)
    v = rng.normal(size=(n_chains, n_part))
    v *= np.sqrt(n_part / np.sum(v * v, axis=1, keepdims=True))
    for _ in range(SCALE["burn"]):
        step_sphere(v, rule, rng)
    pool = []
    for _ in range(SCALE["collect"]):
        for _ in range(SCALE["gap"]):
            step_sphere(v, rule, rng)
        pool.append(v.copy().ravel())
    return np.concatenate(pool)


def moments(s: np.ndarray):
    """Return (m4, m6, P(|v| > 2)) for a pooled sample."""
    s2 = s * s
    return (float(np.mean(s2 * s2)), float(np.mean(s2 * s2 * s2)),
            float(np.mean(np.abs(s) > 2.0)))

## Experiment: Figure 4 — the condensate (§6)

Run both arms under both seeds. The control is reported first, because it is
what licenses reading the sorted arm at all.

In [ ]:
# ── Run both arms  (§6, Figure 4) ────────────────────────────────────────────
data, per_seed = {}, {}
for rule in ("sorted", "random"):
    parts = []
    for sd in SEEDS:
        s = sample(rule, sd)
        per_seed[(rule, sd)] = moments(s)
        parts.append(s)
    data[rule] = np.concatenate(parts)

N = SCALE["n_part"]
m4_eq, m6_eq = sphere_moment(N, 2), sphere_moment(N, 3)

print(f"{'arm':22s} {'seed':>10s} {'m4':>10s} {'m6':>14s} {'P(|v|>2)':>10s}")
for rule in ("random", "sorted"):
    for sd in SEEDS:
        m4, m6, t2 = per_seed[(rule, sd)]
        label = "random pairing (null)" if rule == "random" else "sorted rule"
        print(f"{label:22s} {sd:>10d} {m4:>10.4g} {m6:>14.6g} {t2:>10.5f}")
print(f"{'equilibrium, exact':22s} {'--':>10s} {m4_eq:>10.4g} {m6_eq:>14.6g} {TAIL2_EXACT:>10.5f}")

# The control is the gate. If it misses the equilibrium, stop reading here.
ctrl = [per_seed[("random", sd)] for sd in SEEDS]
ctrl_ok = all(abs(c[0] - m4_eq) / m4_eq < 0.05 and abs(c[2] - TAIL2_EXACT) < 0.005
              for c in ctrl)
print(f"\nNull control returns the equilibrium: {'YES' if ctrl_ok else 'NO'}")
if not ctrl_ok:
    print("  The control has missed. Nothing below can be attributed to the rule.")

In [ ]:
# ── Panel data and the figure  (§6, Figure 4) ────────────────────────────────
edges = np.linspace(-4.0, 4.0, 161)
ctr = 0.5 * (edges[:-1] + edges[1:])
d_sorted, _ = np.histogram(data["sorted"], bins=edges, density=True)
d_random, _ = np.histogram(data["random"], bins=edges, density=True)
d_exact = np.exp(-0.5 * ctr ** 2) / np.sqrt(2.0 * np.pi)

grid = np.logspace(np.log10(0.05), np.log10(60.0), 220)


def survival(s: np.ndarray, g: np.ndarray) -> np.ndarray:
    """Empirical P(|v| > x) on the grid g."""
    a = np.sort(np.abs(s))
    return 1.0 - np.searchsorted(a, g, side="right") / a.size


s_sorted, s_random = survival(data["sorted"], grid), survival(data["random"], grid)
s_exact = np.array([erfc(x / sqrt(2.0)) for x in grid])
# An empirical survival function that has run out of samples reads exactly zero.
# On a log axis that draws a cliff which is an artefact of the sample size, so
# mask it rather than draw it.
m_sorted = np.where(s_sorted > 0, s_sorted, np.nan)
m_random = np.where(s_random > 0, s_random, np.nan)

RED, BLUE, GREY = "#d62728", "#1f77b4", "grey"

# The manuscript figure is drawn under matplotlib's DEFAULT rcParams, not the
# notebook style set above. Reproducing it exactly therefore means stepping
# outside that style for this one figure; otherwise the fonts and line widths
# differ from the published PDF and the two drift apart.
with mpl.rc_context(mpl.rcParamsDefault):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.2, 3.5))

    ax1.semilogy(ctr, d_exact, "-", color=GREY, lw=2.2, label="Maxwellian, exact")
    ax1.semilogy(ctr, d_random, "--", color=BLUE, lw=1.6, label="random pairing, null control")
    ax1.semilogy(ctr, d_sorted, "-", color=RED, lw=1.8, label="sorted rule")
    ax1.set_xlabel(r"$v$"); ax1.set_ylabel("stationary density")
    ax1.set_xlim(-4.0, 4.0); ax1.set_ylim(1e-4, 3e1)
    ax1.legend(loc="upper left", fontsize=7.5, framealpha=0.9)
    ax1.set_title("(a) the condensate", fontsize=10)

    ax2.loglog(grid, s_exact, "-", color=GREY, lw=2.2, label="Maxwellian, exact")
    ax2.loglog(grid, m_random, "--", color=BLUE, lw=1.6, label="random pairing, null control")
    ax2.loglog(grid, m_sorted, "-", color=RED, lw=1.8, label="sorted rule")
    ax2.axhline(2.0 / N, color=RED, lw=0.8, ls=":", alpha=0.7)
    ax2.axvline(np.sqrt(N), color=RED, lw=0.8, ls=":", alpha=0.7)
    ax2.text(0.075, 2.0 / N * 1.5, "$2/N$", color=RED, fontsize=8)
    ax2.text(np.sqrt(N) * 0.30, 2.2e-7, r"$\sqrt{N}$", color=RED, fontsize=8)
    ax2.set_xlabel(r"$x$"); ax2.set_ylabel(r"$P(|v| > x)$")
    ax2.set_xlim(0.05, 60.0); ax2.set_ylim(1e-7, 1.5)
    ax2.legend(loc="lower left", fontsize=7.5, framealpha=0.9)
    ax2.set_title("(b) two particles hold the energy", fontsize=10)

    for ax in (ax1, ax2):
        ax.grid(True, which="major", alpha=0.25)
    fig.tight_layout()
    fig.savefig("fig04_condensate.pdf")
    fig.savefig("fig04_condensate.png", dpi=150)
plt.show()

In [ ]:
# ── Diagnostic analysis ───────────────────────────────────────────────────────
# Three readings the figure is used to support, each checked rather than asserted.
print("=" * 65)
print("DIAGNOSTICS")
print("=" * 65)

srt = data["sorted"]
m4_s = float(np.mean(srt ** 4)); m6_s = float(np.mean(srt ** 6))
tail_s = float(np.mean(np.abs(srt) > 2.0))

print(f"1. Tail deficit and moment inflation coexist.")
print(f"   two-sigma mass retained : {100 * tail_s / TAIL2_EXACT:5.1f}% of equilibrium")
print(f"   m4 inflation            : x{m4_s / m4_eq:,.0f}")
print(f"   m6 inflation            : x{m6_s / m6_eq:,.0f}")
print(f"   The tail is thinner near the shoulder and far heavier past |v| ~ 3.")

print(f"\n2. The plateau sits near 2/N.")
mid = (grid > 0.5) & (grid < 10.0)
print(f"   median survival on 0.5 < x < 10 : {np.nanmedian(m_sorted[mid]):.6f}")
print(f"   2/N                             : {2.0 / N:.6f}")

print(f"\n3. The support reaches sqrt(N) and stops.")
print(f"   max |v| observed : {np.abs(srt).max():.3f}")
print(f"   sqrt(N)          : {np.sqrt(N):.3f}")
print(f"   ratio            : {np.abs(srt).max() / np.sqrt(N):.6f}")

print("\nRead together: about two particles in N are moving at any threshold")
print("between 0.05 and sqrt(N), and between them they carry the whole energy.")
print("Note that 'two particles' is a reading of the plateau, which is flat over")
print("that range. It is not a threshold-free particle count, and the manuscript")
print("does not claim one.")
print("=" * 65)

## Summary

The null control reproduces the exact equilibrium on the sphere, so the sorted
arm can be read as signal rather than as a defect of the sampler. Under the
sorted rule the stationary state is a condensate: almost all mass sits at the
origin, the survival function holds a plateau near $2/N$ out to $|v|=\sqrt{N}$,
and the fourth and sixth moments are inflated by large factors while the
two-sigma mass is *depleted*. Those two readings are the same fact seen from
two directions, not competing diagnoses.

Set `FULL = True` in the setup cell and re-run to reproduce the manuscript
figure. The PDF this produces is byte-identical to the published
`fig04_condensate.pdf` when the environment variable `SOURCE_DATE_EPOCH` is
pinned, since matplotlib otherwise stamps a creation date into the file. At
`FULL = False` the shape is already correct; only the wings of panel (a) are
ragged, because they are estimated from fewer samples.

Note that the figure cell steps outside the notebook's plot style and draws
under matplotlib's defaults. That is deliberate: the published figure was drawn
that way, and matching it matters more here than house style does.

**What this notebook does not settle.** It describes the stationary law at one
value of $N$ without identifying it. Section 9 of the manuscript takes that
question up.

In [ ]:
# ── Self-contained HTML report ────────────────────────────────────────────────
def fig_to_base64(f: plt.Figure) -> str:
    """Encode a Figure as a base64 PNG string for inline HTML embedding."""
    buf = io.BytesIO()
    f.savefig(buf, format='png', dpi=130, bbox_inches='tight')
    buf.seek(0)
    return base64.b64encode(buf.read()).decode('utf-8')


img = fig_to_base64(fig)
plt.close(fig)

rows = "".join(
    f"<tr><td>{'random pairing (null)' if r == 'random' else 'sorted rule'}</td>"
    f"<td>{sd}</td><td>{per_seed[(r, sd)][0]:.4g}</td>"
    f"<td>{per_seed[(r, sd)][1]:.6g}</td><td>{per_seed[(r, sd)][2]:.5f}</td></tr>"
    for r in ("random", "sorted") for sd in SEEDS)

html = f"""<!DOCTYPE html><html><head><meta charset="utf-8">
<title>Figure 4 &mdash; the condensate</title>
<style>
 body {{ font-family: Georgia, serif; margin: 2em auto; max-width: 60em; color:#222; }}
 table {{ border-collapse: collapse; margin: 1em 0; }}
 th, td {{ border: 1px solid #bbb; padding: 5px 12px; text-align: right; }}
 th:first-child, td:first-child {{ text-align: left; }}
 .meta {{ background:#f4f4f4; padding:.6em 1em; font-size:.9em; }}
</style></head><body>
<h1>Figure 4 &mdash; the condensate of the sorted Kac rule</h1>
<div class="meta">Generated {datetime.now():%Y-%m-%d %H:%M} &middot;
 mode {'FULL' if FULL else 'QUICK'} &middot; N = {N} &middot;
 chains {SCALE['n_chains']} &middot; seeds {SEEDS[0]}, {SEEDS[1]}</div>
<img src="data:image/png;base64,{img}" style="width:100%">
<h2>Measurements</h2>
<table><tr><th>arm</th><th>seed</th><th>m4</th><th>m6</th><th>P(|v|&gt;2)</th></tr>
{rows}
<tr><td><em>equilibrium, exact</em></td><td>&mdash;</td><td>{m4_eq:.4g}</td>
<td>{m6_eq:.6g}</td><td>{TAIL2_EXACT:.5f}</td></tr></table>
<h2>Reading</h2>
<p>The null control returns the exact equilibrium, which is what licenses the
sorted arm. The sorted rule retains {100 * tail_s / TAIL2_EXACT:.1f}% of the
equilibrium's two-sigma mass while inflating m4 by a factor of
{m4_s / m4_eq:,.0f} and m6 by {m6_s / m6_eq:,.0f}. Depleted shoulder and
inflated moments are the same condensate seen from two directions.</p>
<p style="color:#666;font-size:.9em">Companion text: J. M. Hyman,
<em>A velocity-sorting obstruction for Array-RQMC</em>, &sect;6, Figure 4.</p>
</body></html>"""

with open("Figure4_Report.html", "w", encoding="utf-8") as fh:
    fh.write(html)
print(f"Figure4_Report.html written ({len(html) // 1024} KB)")

In [ ]:
# ── Download outputs ──────────────────────────────────────────────────────────
output_files = ['Figure4_Report.html', 'fig04_condensate.pdf', 'fig04_condensate.png']
try:
    from google.colab import files
    for fname in output_files:
        files.download(fname)
    print("Downloads triggered.")
except ImportError:
    import os
    print("Not in Colab — files saved locally:")
    for fname in output_files:
        print(f"  {fname}  ({os.path.getsize(fname) // 1024} KB)")